# Week 41, Day 3: Two pointers

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w041d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
nums = [1, 1, 2, 3, 3, 3, 7, 7, 9]
write = 1
for read in range(1, len(nums)):
    if nums[read] != nums[write - 1]:
        nums[write] = nums[read]
        write += 1
print(write, nums[:write], nums)

In [ ]:
def pair_sum(nums, target):
    left, right = 0, len(nums) - 1
    steps = 0
    while left < right:
        steps += 1
        total = nums[left] + nums[right]
        if total == target:
            return (nums[left], nums[right]), steps
        if total < target:
            left += 1
        else:
            right -= 1
    return None, steps

nums = list(range(0, 2000, 3))          # 667 sorted numbers
print(pair_sum(nums, 1999))
print(pair_sum(nums, 2000))

In [ ]:
def three_sum(nums):
    nums = sorted(nums)
    found = []
    for i in range(len(nums) - 2):
        if i > 0 and nums[i] == nums[i - 1]:
            continue                                   # same first number as last time
        left, right = i + 1, len(nums) - 1
        while left < right:
            total = nums[i] + nums[left] + nums[right]
            if total < 0:
                left += 1
            elif total > 0:
                right -= 1
            else:
                found.append((nums[i], nums[left], nums[right]))
                left += 1
                while left < right and nums[left] == nums[left - 1]:
                    left += 1                          # skip repeats of the middle number
    return found

print(three_sum([-1, 0, 1, 2, -1, -4, 2, -2]))

In [ ]:
def most_water(heights):
    left, right = 0, len(heights) - 1
    best = 0
    while left < right:
        width = right - left
        best = max(best, width * min(heights[left], heights[right]))
        if heights[left] < heights[right]:
            left += 1          # the shorter wall can't do better with any closer partner
        else:
            right -= 1
    return best

print(most_water([1, 8, 6, 2, 5, 4, 8, 3, 7]))

In [ ]:
def move_zeros(nums):
    """Move every 0 to the end, keeping the order of the rest, in place."""
    write = 0
    for read in range(len(nums)):
        if nums[read] != 0:
            nums[write], nums[read] = nums[read], nums[write]
            write += 1

data = [0, 3, 0, 0, 7, 1, 0, 4]
move_zeros(data)
print(data)

In [ ]:
def three_way(colours):
    """Sort a list of 0s, 1s and 2s in place, in one pass."""
    low, mid, high = 0, 0, len(colours) - 1
    while mid <= high:
        if colours[mid] == 0:
            colours[low], colours[mid] = colours[mid], colours[low]
            low += 1
            mid += 1
        elif colours[mid] == 1:
            mid += 1
        else:
            colours[mid], colours[high] = colours[high], colours[mid]
            high -= 1                  # don't move mid: the swapped-in item is still unknown

flags = [2, 0, 1, 2, 1, 0, 0, 2, 1]
three_way(flags)
print(flags)

In [ ]:
def common(a, b):
    """The values in both sorted lists (with repeats as often as both have them)."""
    i = j = 0
    out = []
    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            out.append(a[i])
            i += 1
            j += 1
        elif a[i] < b[j]:
            i += 1
        else:
            j += 1
    return out

def is_subsequence(small, big):
    """Do small's items appear in big, in order (not necessarily side by side)?"""
    i = 0
    for item in big:
        if i < len(small) and small[i] == item:
            i += 1
    return i == len(small)

print(common([1, 2, 2, 3, 5, 8], [2, 2, 2, 5, 9]))
print(is_subsequence("ace", "abcde"), is_subsequence("aec", "abcde"))

> **Key idea.** Two pointers turn many O(n²) problems into one O(n) pass. Closing in from both ends works on sorted data and symmetric problems, and is safe when each move rules out one end because no remaining partner could work with it. A reader and a writer rearrange a list in place: `read` visits everything, `write` marks where the next keeper goes. One pointer in each of two sorted sequences merges, intersects or matches them. Sorting first (O(n log n)) often makes a problem ready for two pointers.

### Exercise: Duplicates, in place

Write `dedupe(nums)` for a **sorted** list: rearrange it in place so its first `k` items are the distinct values in order, and return `k`. Use a reader and a writer, no extra list and no `set`.

In [ ]:
def dedupe(nums):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_dedupe():\n    \"in place\"\n    for nums, want in [([1, 1, 2, 3, 3], [1, 2, 3]), ([], []), ([4], [4]), ([2, 2, 2], [2]), ([-3, -1, -1, 0, 5, 5, 5, 9], [-3, -1, 0, 5, 9])]:\n        before = len(nums)\n        k = dedupe(nums)\n        assert nums[:k] == want and len(nums) == before, f\"got {nums[:k]}, k={k}\"\n    assert \"set(\" not in source")

### Exercise: Sorted squares

Write `sorted_squares(nums)`: given a list sorted in ascending order (it may contain negatives), return a new list of the squares, in ascending order, in **O(n)**, without sorting. The biggest square is always at one of the two ends.

In [ ]:
def sorted_squares(nums):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_squares():\n    \"squares\"\n    assert sorted_squares([-7, -3, 2, 3, 11]) == [4, 9, 9, 49, 121] and sorted_squares([-4, -1, 0, 3, 10]) == [0, 1, 9, 16, 100]\n    assert sorted_squares([]) == [] and sorted_squares([-2]) == [4] and sorted_squares([1, 2]) == [1, 4]\n    import ast\n    calls = [n.func for n in ast.walk(ast.parse(source)) if isinstance(n, ast.Call)]\n    assert not any(getattr(f, \"id\", None) == \"sorted\" or getattr(f, \"attr\", None) == \"sort\" for f in calls), \"no sorting\"")

### Exercise: The closest pair

Write `closest_pair(nums, target)` for a sorted list of at least two numbers: return the pair `(a, b)` (with `a` before `b` in the list) whose sum is closest to `target`. On a tie, prefer the pair with the **smaller** sum. Use two pointers closing in, in O(n).

In [ ]:
def closest_pair(nums, target):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_closest():\n    \"closest\"\n    assert closest_pair([1, 3, 4, 7, 10], 15) == (4, 10) and closest_pair([1, 3, 4, 7, 10], 9) == (1, 7)\n    assert closest_pair([1, 3, 4, 7, 10], 100) == (7, 10) and closest_pair([1, 3, 4, 7, 10], -5) == (1, 3)\n    assert closest_pair([1, 4, 6, 9], 12) == (4, 9) and closest_pair([2, 5], 0) == (2, 5)\n    assert closest_pair([1, 2, 4, 5], 8) == (2, 5), \"7 and 9 are both 1 away: prefer 7\"")

### Exercise: The most water

Write `most_water(heights)`: the largest amount of water two walls can hold, where walls `i < j` hold `(j - i) * min(heights[i], heights[j])`. Return `(amount, i, j)` for the best pair; on a tie, any best pair will do for `i` and `j`, but the amount must be right. It must run in O(n): the checks use 20,000 walls. Fewer than two walls hold `(0, None, None)`.

In [ ]:
def most_water(heights):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef brute(h):\n    return max(((j - i) * min(h[i], h[j]) for i in range(len(h)) for j in range(i + 1, len(h))), default=0)\ndef test_small():\n    \"small cases\"\n    for h in [[1, 8, 6, 2, 5, 4, 8, 3, 7], [1, 1], [4, 3, 2, 1, 4], [1, 2, 1], [2, 3, 10, 5, 7, 8, 9]]:\n        amount, i, j = most_water(h)\n        assert amount == brute(h) and (j - i) * min(h[i], h[j]) == amount, h\n    assert most_water([5]) == (0, None, None) and most_water([]) == (0, None, None)\ndef test_big():\n    \"20,000 walls\"\n    rng = random.Random(7)\n    h = [rng.randint(1, 10_000) for _ in range(20_000)]\n    amount, i, j = most_water(h)\n    assert (j - i) * min(h[i], h[j]) == amount and amount >= max((k - 0) * min(h[0], h[k]) for k in range(1, len(h)))")

### Exercise: Three kinds in one pass

Write `partition3(items, pivot)`, which rearranges a list in place so that every item **less than** `pivot` comes first, then every item **equal** to it, then every item **greater**, in a single pass with three pointers (Dijkstra's Dutch national flag method). Return `(lt, gt)`: the index where the equal items start and the index just after they end. The order within each group doesn't matter.

In [ ]:
def partition3(items, pivot):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_partition():\n    \"partitioning\"\n    rng = random.Random(3)\n    for _ in range(200):\n        items = [rng.randint(0, 6) for _ in range(rng.randint(0, 15))]\n        original = sorted(items)\n        lt, gt = partition3(items, 3)\n        assert sorted(items) == original, \"same items\"\n        assert all(x < 3 for x in items[:lt]) and all(x == 3 for x in items[lt:gt]) and all(x > 3 for x in items[gt:]), items\n    import ast\n    assert not any(getattr(n.func, \"attr\", None) == \"sort\" for n in ast.walk(ast.parse(source)) if isinstance(n, ast.Call)), \"no sorting\"")

### Exercise: Merging in place, from the back

`a` has `m` sorted numbers followed by `n` spare slots (holding `None`); `b` has `n` sorted numbers. Write `merge_into(a, m, b)`, which merges `b` into `a` **in place** so `a` is fully sorted, without making another list. Fill `a` from the back, with three pointers: the last real item of `a`, the last of `b`, and the slot to write.

In [ ]:
def merge_into(a, m, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_merge():\n    \"merging\"\n    a = [1, 4, 7, None, None, None]\n    merge_into(a, 3, [2, 5, 9])\n    assert a == [1, 2, 4, 5, 7, 9]\n    a = [None, None]\n    merge_into(a, 0, [3, 8])\n    assert a == [3, 8]\n    a = [5, 6]\n    merge_into(a, 2, [])\n    assert a == [5, 6]\n    a = [10, 20, None]\n    merge_into(a, 2, [1])\n    assert a == [1, 10, 20]")

### Bug hunt: The wrong pointer moves

When the sum is too small, this moves `right`, which makes it even smaller, so it misses `(3, 8)` and prints `None`. Fix it so it prints `(3, 8)`.

In [ ]:
def pair_sum(nums, target):
    left, right = 0, len(nums) - 1
    while left < right:
        total = nums[left] + nums[right]
        if total == target:
            return nums[left], nums[right]
        if total < target:
            right -= 1
        else:
            left += 1
    return None

print(pair_sum([1, 3, 5, 8, 12], 11))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"move the right pointer\"\n    expect_output(\"(3, 8)\")")

### Bug hunt: Running off the end

This palindrome check skips anything that isn't a letter or digit, but its inner skipping loops don't check that the pointers are still in range, so a string of punctuation crashes it with `IndexError`. Fix it so this prints `True`, `False`, `True`.

In [ ]:
def is_palindrome(text):
    left, right = 0, len(text) - 1
    while left < right:
        while not text[left].isalnum():
            left += 1
        while not text[right].isalnum():
            right -= 1
        if text[left].lower() != text[right].lower():
            return False
        left += 1
        right -= 1
    return True

print(is_palindrome("Never odd, or even!"))
print(is_palindrome("Python"))
print(is_palindrome("?!..,"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"stay in range\"\n    expect_output(\"True\\nFalse\\nTrue\")\n    assert is_palindrome(\"a.\") and is_palindrome(\"\") and not is_palindrome(\"ab!\")")

### Bug hunt: Zeros that multiplied

`move_zeros` copies each non-zero item forward, but never puts the zeros back at the end, so the old values are left behind: this prints `[3, 7, 1, 7, 1]`. Fix it (in place, keeping the order of the non-zero items) so it prints `[3, 7, 1, 0, 0]`.

In [ ]:
def move_zeros(nums):
    write = 0
    for read in range(len(nums)):
        if nums[read] != 0:
            nums[write] = nums[read]
            write += 1

data = [0, 3, 0, 7, 1]
move_zeros(data)
print(data)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"zeros at the end\"\n    expect_output(\"[3, 7, 1, 0, 0]\")\n    xs = [1, 0, 2, 0, 0, 3]\n    move_zeros(xs)\n    assert xs == [1, 2, 3, 0, 0, 0]")

### Puzzle: Is it a subsequence?

Arrange the lines into a function that checks whether `small`'s items appear in `big` in order, with one pointer in each. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# Answer without running code.

p = ___   # the extra space (beyond the input) used by a reader-and-writer dedupe: "O(1)" or "O(n)"
q = ___   # the time of three_sum with sorting and two pointers: "O(n)", "O(n log n)", "O(n^2)" or "O(n^3)"
r = ___   # most loop steps pair_sum takes on a sorted list of 100 numbers
s = ___   # which wall most_water moves when the heights at the two ends are 3 and 9: "3" or "9"
t = ___   # True or False: closing-in pair_sum works on an unsorted list

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"O(1)\" and q == \"O(n^2)\" and r == 99, \"p, q and r\"\n    assert s == \"3\" and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: trapped rain

A row of walls has heights `heights`. After rain, water fills every dip: above each position, the water level is the **smaller** of the tallest wall to its left (including itself) and the tallest wall to its right (including itself), and the water there is that level minus the wall's height. Write `trapped(heights)`, the total water, in **O(n) time and O(1) extra space**, with two pointers closing in. The checks use 50,000 walls, and compare with a simple version.

In [ ]:
def trapped(heights):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef simple(h):\n    return sum(min(max(h[:i + 1]), max(h[i:])) - h[i] for i in range(len(h)))\ndef test_small():\n    \"small cases\"\n    for h in [[0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1], [4, 2, 0, 3, 2, 5], [], [3], [1, 2, 3], [3, 2, 1], [2, 0, 2], [5, 0, 0, 0, 5, 0, 1]]:\n        assert trapped(h) == simple(h), (h, trapped(h), simple(h))\n    rng = random.Random(1)\n    for _ in range(200):\n        h = [rng.randint(0, 9) for _ in range(rng.randint(0, 20))]\n        assert trapped(h) == simple(h), h\ndef test_big():\n    \"50,000 walls, quickly\"\n    rng = random.Random(2)\n    h = [rng.randint(0, 1000) for _ in range(50_000)]\n    total = trapped(h)\n    assert isinstance(total, int) and total > 0")

## Warm-up

One pass, no extra list: `read` looks at every item, `write` marks where the next keeper goes. The items after `write` are leftovers nobody needs. This is the "reader and writer" shape, one of three you'll meet today.

## Three shapes

| Shape | Pointers | Typical problems |
|---|---|---|
| **Closing in** | `left` at the start, `right` at the end, moving towards each other | pairs with a sum in a sorted list, palindromes, reversing, the most water between walls |
| **Reader and writer** (or fast and slow) | both start at the left; the reader moves every step, the writer only when something is kept | removing duplicates or zeros in place, partitioning, compacting |
| **One in each sequence** | `i` in one list, `j` in another | merging sorted lists (Week 19's merge sort), intersections, "is this a subsequence?" |

The loop always ends when the pointers meet or one runs off the end, and each step moves at least one pointer forward: so the whole thing is **O(n)**.

## Closing in from both ends

The classic: in a **sorted** list, find two numbers that add up to a target. Trying every pair is O(n²). With two pointers, compare the sum of the ends with the target: too small, move `left` right (to a bigger number); too big, move `right` left.

A harder relative: **three** numbers that add up to 0. Sort the list (O(n log n)), then, for each number, find pairs after it summing to its negative with two pointers: O(n²) in all, instead of O(n³) for every triple. Skipping repeated values keeps each triple once:

## Why it never misses

It can seem too good to be true: each step throws away a whole row of pairs without checking them. Here's why that's safe, for the pair sum.

Suppose `nums[left] + nums[right]` is **too small**. Then `nums[left]` plus **any** number at or before `right` is too small as well, because those numbers are no bigger than `nums[right]` (the list is sorted). So `nums[left]` can't be in any pair still possible, and moving `left` past it loses nothing. The same argument, mirrored, covers moving `right` when the sum is too big.

That's the shape of every closing-in proof: **each move rules out one end, because no remaining partner could work with it.** If you can't make that argument for a problem, two pointers probably isn't the answer, which is worth checking before you start.

The "most water" problem shows a less obvious version. Given wall heights, pick two walls to hold the most water between them (the width times the **shorter** wall):

Why move the **shorter** wall? Any container using it with a closer partner is narrower, and its height is still capped by that short wall: so it can only hold less. The shorter wall is finished; move past it.

## A reader and a writer

To change a list **in place** (no second list, O(1) extra space), let a `read` pointer visit every item while a `write` pointer marks where the next item to keep should go. Everything before `write` is the finished result.

A three-way version sorts a list of only three kinds of values in one pass. It's known as the **Dutch national flag** problem, after the three-coloured flag, from Edsger Dijkstra's *A Discipline of Programming* (1976). Three pointers split the list into four regions: done-low, done-middle, unknown, and done-high:

The same idea is the heart of quicksort's partition step (Week 19).

## Two sequences

With one pointer in each of two **sorted** sequences, you can walk them together, always moving the one that's behind:

Each step moves at least one pointer, so the walk is O(len(a) + len(b)), where checking every item of one list against the other would be O(len(a) × len(b)).

## Practice

## Quiz

## Daily challenge

## Before you go

For each of the three shapes, write the skeleton from memory: the pointers, the loop condition, and what moves each one. Tomorrow: sliding windows, the two-pointer pattern for "every run of neighbouring items".